In [ ]:
pip install emoji #cell 1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd #cell 2
import re
import emoji
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
from transformers import BertTokenizer, pipeline
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
from tabulate import tabulate

# Load the dataset
df = pd.read_csv("/content/preprocessed_dataset (1) (1).csv")

# Define label columns
label_columns = ["sarcasm", "joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]

# Ensure all required columns exist in dataset
for col in label_columns:
    if col not in df.columns:
        df[col] = 0

df[label_columns] = df[label_columns].fillna(0)

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = emoji.demojize(text, delimiters=(" ", " "))  # Convert emojis to text labels
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    return text

df["Tweet"] = df["Tweet"].apply(preprocess_text)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df["Tweet"], df[label_columns], test_size=0.2, random_state=42
)

# Reduce TF-IDF feature size
vectorizer = TfidfVectorizer(max_features=2000, stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train SVM model with OneVsRestClassifier
svm_model = OneVsRestClassifier(SVC(kernel='linear', probability=True))
svm_model.fit(X_train_tfidf, y_train)

# Train Random Forest model with fewer estimators
rf_model = OneVsRestClassifier(RandomForestClassifier(n_estimators=50, random_state=42))
rf_model.fit(X_train_tfidf, y_train)


OneVsRestClassifier(estimator=RandomForestClassifier(n_estimators=50,
                                                     random_state=42))

In [ ]:
from sklearn.metrics import classification_report

# SVM evaluation
svm_pred = svm_model.predict(X_test_tfidf)
print("=== SVM Performance on Test Data ===")
print(classification_report(y_test, svm_pred, target_names=label_columns, zero_division=0))

# Random Forest evaluation
rf_pred = rf_model.predict(X_test_tfidf)
print("=== Random Forest Performance on Test Data ===")
print(classification_report(y_test, rf_pred, target_names=label_columns, zero_division=0))

=== SVM Performance on Test Data ===
              precision    recall  f1-score   support

     sarcasm       1.00      0.99      0.99       411
         joy       0.79      0.54      0.64       205
       trust       0.50      0.03      0.06        29
        fear       0.82      0.44      0.57       103
    surprise       0.67      0.16      0.26        25
     sadness       0.75      0.25      0.37       188
     disgust       0.68      0.47      0.55       214
       anger       0.74      0.53      0.62       214
anticipation       0.00      0.00      0.00        86

   micro avg       0.85      0.56      0.68      1475
   macro avg       0.66      0.38      0.45      1475
weighted avg       0.77      0.56      0.63      1475
 samples avg       0.71      0.65      0.66      1475

=== Random Forest Performance on Test Data ===
              precision    recall  f1-score   support

     sarcasm       1.00      0.99      0.99       411
         joy       0.75      0.56      0.64     

In [ ]:
# Load a Pretrained Emotion Detection Model
emotion_classifier = pipeline("text-classification", model="joeddav/distilbert-base-uncased-go-emotions-student", top_k=None)

def predict_emotion(text):
    text = preprocess_text(text)
    tfidf_features = vectorizer.transform([text])
    svm_pred = svm_model.predict(tfidf_features)
    rf_pred = rf_model.predict(tfidf_features)

    bert_pred = emotion_classifier(text)[0] # Use pretrained model for BERT predictions

    # Filter out low confidence emotions and normalize
    filtered_bert_pred = {entry['label']: entry['score'] for entry in bert_pred if entry['score'] > 0.01}
    total_score = sum(filtered_bert_pred.values())
    normalized_bert_pred = {k: round((v / total_score) * 100, 2) for k, v in filtered_bert_pred.items()}

    # Keep only top 3 emotions
    sorted_bert_pred = dict(sorted(normalized_bert_pred.items(), key=lambda item: item[1], reverse=True)[:3])

    return {
        "SVM": svm_pred.tolist(),
        "Random Forest": rf_pred.tolist(),
        "BERT": sorted_bert_pred
    }


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
# User Input for Evaluation
while True:
    user_text = input("Enter a sentence to analyze emotions (or type 'exit' to quit): ")
    if user_text.lower() == 'exit':
        break
    result = predict_emotion(user_text)
    print("\nPredicted Emotions (BERT Model):")
    print(tabulate(result["BERT"].items(), headers=["Emotion", "Confidence (%)"], tablefmt="grid"))



Predicted Emotions (BERT Model):
+-------------+------------------+
| Emotion     |   Confidence (%) |
+=============+==================+
| fear        |            40.62 |
+-------------+------------------+
| disgust     |            13.59 |
+-------------+------------------+
| disapproval |             6.87 |
+-------------+------------------+


KeyboardInterrupt: Interrupted by user